# HF-RISK Colab Runner

Run this notebook with the **Google Colab VS Code extension**. The normal VS Code PowerShell terminal still uses your laptop; these notebook cells use the selected Colab runtime when the kernel/runtime is set to Colab.

## No Google Drive workflow

If Colab cannot see your local project files, upload `hf_risk_colab_bundle.zip` to the Colab server using the VS Code Colab extension. In VS Code: right-click the ZIP or use the Command Palette command `Colab: Upload to Colab` / `Upload to Colab`. The next cell will unzip it into `/content/MSc Project`.

Pipeline order: `02_preprocessing.py -> 03_modelling.py -> 04_survival.py -> 05_shap.py -> 08_external_validation_mimic.py -> 10_advanced_evaluation.py`.


In [9]:
# 1) Confirm this is running on Colab, not local Windows.
import os, sys, platform

print('Python:', sys.version)
print('Executable:', sys.executable)
print('Platform:', platform.platform())
print('Current folder:', os.getcwd())

runtime_os = os.environ.get('OS', os.name).lower()
if runtime_os == 'nt' or runtime_os.startswith('windows'):
    print('WARNING: This kernel is Windows/local. Select a Colab runtime in VS Code before running the heavy scripts.')
else:
    print('OK: Non-Windows runtime detected. This should be Colab/Linux if you selected Colab.')

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Executable: /usr/bin/python3
Platform: Linux-6.6.113+-x86_64-with-glibc2.35
Current folder: /content
OK: Non-Windows runtime detected. This should be Colab/Linux if you selected Colab.


In [13]:
# 2) Locate the MSc Project folder in the Colab runtime.
# If you uploaded hf_risk_colab_bundle.zip via the Colab VS Code extension, this cell unzips it first.
from pathlib import Path
import os, zipfile

def looks_like_project(folder: Path) -> bool:
    return (folder / '02_preprocessing.py').exists() and (folder / '03_modelling.py').exists()

# Look for the uploaded bundle in common places.
bundle_candidates = [
    Path('/content/hf_risk_colab_bundle.zip'),
    Path.cwd() / 'hf_risk_colab_bundle.zip',
]
if Path('/content').exists():
    try:
        for p in Path('/content').rglob('hf_risk_colab_bundle.zip'):
            bundle_candidates.append(p)
            if len(bundle_candidates) >= 8:
                break
    except Exception:
        pass

bundle = next((b for b in bundle_candidates if b.exists()), None)
if bundle is not None:
    extract_root = Path('/content') if Path('/content').exists() else Path.cwd()
    with zipfile.ZipFile(bundle, 'r') as zf:
        zf.extractall(extract_root)
    print('Unzipped bundle:', bundle, '->', extract_root)

# Check common project locations first.
candidates = [
    Path.cwd(),
    Path.cwd() / 'MSc Project',
    Path('/content/MSc Project'),
    Path('/content/auto-annotated-portfolio-9bdeb/MSc Project'),
    Path('/content/auto-annotated-portfolio-9bdeb'),
    Path('/content/drive/MyDrive/MSc Project'),
    Path('/content/drive/MyDrive/auto-annotated-portfolio-9bdeb/MSc Project'),
]

project_dir = next((cand for cand in candidates if looks_like_project(cand)), None)

# Fallback: search for the script and accept its parent as project root.
if project_dir is None and Path('/content').exists():
    try:
        for hit in Path('/content').rglob('02_preprocessing.py'):
            cand = hit.parent
            if looks_like_project(cand):
                project_dir = cand
                break
    except Exception:
        pass

if project_dir is None:
    print('Current folder:', Path.cwd())
    if Path('/content').exists():
        print('/content sample:', [p.name for p in list(Path('/content').iterdir())[:20]])
    raise FileNotFoundError(
        'Could not find MSc Project scripts. Upload hf_risk_colab_bundle.zip to the Colab server '
        'or open this notebook from a workspace that already contains the project.'
)

os.chdir(project_dir)
print('Using project folder:', project_dir)
print('Files here:', [p.name for p in Path.cwd().iterdir() if p.name.endswith('.py')][:12])

Unzipped bundle: /content/hf_risk_colab_bundle.zip -> /content
Using project folder: /content
Files here: ['04_survival.py', '03_modelling.py', '10_advanced_evaluation.py', '08_external_validation_mimic.py', '02_preprocessing.py', '05_shap.py']


In [14]:
# 3) Install dependencies into the Colab runtime.
# Runtime installs are temporary; rerun after reconnecting to a fresh Colab VM.
!python -m pip install -q -r requirements.txt

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.1/409.1 kB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 90.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 137.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 140.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 127.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.1/222.1 kB 26.0 MB/s eta 0:00:00


In [15]:
# 4) Colab speed patch: use all available CPU cores on Colab/Linux.
# This reverses the local Windows workaround (n_jobs=1) in the runtime copy.
from pathlib import Path

for file_name in ['03_modelling.py', '04_survival.py']:
    path = Path(file_name)
    text = path.read_text(encoding='utf-8')
    updated = text.replace('n_jobs=1', 'n_jobs=-1')
    if updated != text:
        path.write_text(updated, encoding='utf-8')
        print(f'Updated {file_name}: n_jobs=1 -> n_jobs=-1')
    else:
        print(f'{file_name}: already using n_jobs=-1 or no n_jobs=1 found')

Updated 03_modelling.py: n_jobs=1 -> n_jobs=-1
Updated 04_survival.py: n_jobs=1 -> n_jobs=-1


In [16]:
# 5) Quick leakage check before heavy modelling.
from pathlib import Path

try:
    pd = __import__('pandas')
except Exception as exc:
    raise ImportError("pandas is required. Run cell 4 to install requirements first.") from exc

raw = pd.read_csv('data/dat.csv', nrows=1)
print('Raw dataset columns:', raw.shape[1])
print('Required data present:', Path('data/dat.csv').exists())
print('MIMIC data present:', Path('data/mimic_hf_cohort.csv').exists())
print('Script 10 present:', Path('10_advanced_evaluation.py').exists())

Raw dataset columns: 167
Required data present: True
MIMIC data present: True
Script 10 present: True


In [17]:
# 6) Run the full clean pipeline.
# If a cell times out or disconnects, rerun from the failed script onward.
import os, subprocess, sys, time

scripts = [
    '02_preprocessing.py',
    '03_modelling.py',
    '04_survival.py',
    '05_shap.py',
    '08_external_validation_mimic.py',
    '10_advanced_evaluation.py',
]

env = os.environ.copy()
env['PYTHONIOENCODING'] = 'utf-8'

for script in scripts:
    print('\n' + '=' * 80)
    print(f'RUNNING: {script}')
    print('=' * 80)
    start = time.time()
    result = subprocess.run([sys.executable, script], env=env)
    mins = (time.time() - start) / 60
    print(f'FINISHED: {script} in {mins:.1f} min | exit code={result.returncode}')
    if result.returncode != 0:
        raise RuntimeError(f'{script} failed. Fix that script/output, then rerun this cell from the failed script onward.')


RUNNING: 02_preprocessing.py
FINISHED: 02_preprocessing.py in 0.1 min | exit code=0

RUNNING: 03_modelling.py
FINISHED: 03_modelling.py in 3.2 min | exit code=0

RUNNING: 04_survival.py
FINISHED: 04_survival.py in 0.3 min | exit code=0

RUNNING: 05_shap.py
FINISHED: 05_shap.py in 0.2 min | exit code=0

RUNNING: 08_external_validation_mimic.py
FINISHED: 08_external_validation_mimic.py in 0.4 min | exit code=0

RUNNING: 10_advanced_evaluation.py
FINISHED: 10_advanced_evaluation.py in 0.3 min | exit code=0


In [18]:
# 7) Verification summary after pipeline completes.
import json
from pathlib import Path

try:
    pd = __import__('pandas')
except Exception as exc:
    raise ImportError("pandas is required. Run cell 4 to install requirements first.") from exc

bad_cols = [
    'inpatient.number',
    'dischargeDay',
    'Unnamed: 0',
    'outcome.during.hospitalization',
    're.admission.within.28.days',
    're.admission.within.3.months',
    're.admission.time..days.from.admission.',
    'return.to.emergency.department.within.6.months',
    'time.to.emergency.department.within.6.months',
]

x_test = pd.read_csv('outputs/X_test.csv')
present_bad = [c for c in bad_cols if c in x_test.columns]
print('X_test shape:', x_test.shape)
print('Leakage/ID/date columns still present:', present_bad or 'None')

with open('outputs/best_models_info.json', 'r', encoding='utf-8') as f:
    best = json.load(f)
print('Best model info:')
for k, v in best.items():
    print(f"  {k}: {v.get('name')} | test_auc={v.get('test_auc')}")

adv = Path('outputs/advanced_evaluation')
expected = [
    'calibration_curves.png', 'calibration_metrics.csv',
    'dca_curves.png', 'dca_results.csv',
    'subgroup_gender.png', 'subgroup_gender.csv',
    'subgroup_ckd.png', 'subgroup_ckd.csv',
    'advanced_evaluation_summary.json',
]
print('Advanced evaluation outputs:')
for name in expected:
    print(f'  {name}:', 'OK' if (adv / name).exists() else 'MISSING')

if Path('outputs/external_validation/external_validation_results.csv').exists():
    print('External validation:')
    display(pd.read_csv('outputs/external_validation/external_validation_results.csv'))

if Path('outputs/shap/shap_importance.csv').exists():
    print('Top SHAP features:')
    display(pd.read_csv('outputs/shap/shap_importance.csv').head(12))

X_test shape: (402, 143)
Leakage/ID/date columns still present: None
Best model info:
  28d_death: Random Forest | test_auc=0.7811
  3m_death: Random Forest | test_auc=0.8392
  6m_death: XGBoost | test_auc=0.7098
  6m_readmission: XGBoost | test_auc=0.6497
Advanced evaluation outputs:
  calibration_curves.png: OK
  calibration_metrics.csv: OK
  dca_curves.png: OK
  dca_results.csv: OK
  subgroup_gender.png: OK
  subgroup_gender.csv: OK
  subgroup_ckd.png: OK
  subgroup_ckd.csv: OK
  advanced_evaluation_summary.json: OK
External validation:


,display_name,n,positive_cases,positive_rate,external_auc,ci_low,ci_high,internal_auc
0,28-day Mortality,42990,4419,10.279135,0.523527,0.514996,0.532080,0.7811
1,6-month Mortality,42990,9441,21.960921,0.599474,0.592784,0.605878,0.7098


Top SHAP features:


,feature,mean_abs_shap
0,GCS,0.750208
1,moderate_to_severe_chronic_kidney_disease,0.647674
2,mitral_valve_AMS,0.574778
3,left_ventricular_end_diastolic_diameter_LV,0.388107
4,liver_disease,0.366646
5,congestive_heart_failure,0.357083
6,eye_opening,0.337409
7,reduced_ef_flag,0.265022
8,basophil_ratio,0.258408
9,creatine_kinase,0.231974


## After It Finishes

Bring back or sync these folders/files to your local VS Code workspace if the extension does not automatically sync them:

- `MSc Project/models/`
- `MSc Project/outputs/`
- `MSc Project/03_modelling.py` and `04_survival.py` only if you intentionally want local files to stay at `n_jobs=-1`. For Windows local runs, keep `n_jobs=1`.

In [19]:
import zipfile
from pathlib import Path

project_dir = Path.cwd()
zip_path = Path('/content/hf_risk_final_outputs.zip')

with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for folder_name in ['outputs', 'models']:
        folder = project_dir / folder_name
        if folder.exists():
            for file in folder.rglob('*'):
                if file.is_file():
                    zf.write(file, arcname=str(file.relative_to(project_dir)))

print('Created:', zip_path)
print('Size MB:', round(zip_path.stat().st_size / 1_000_000, 2))


Created: /content/hf_risk_final_outputs.zip
Size MB: 9.03


In [22]:
from pathlib import Path
import pandas as pd, json

checks = [
    'outputs/best_models_info.json',
    'outputs/shap/shap_importance.csv',
    'outputs/external_validation/external_validation_results.csv',
    'outputs/advanced_evaluation/advanced_evaluation_summary.json',
    'outputs/advanced_evaluation/calibration_metrics.csv',
    'outputs/advanced_evaluation/dca_results.csv',
    'outputs/advanced_evaluation/subgroup_gender.csv',
    'outputs/advanced_evaluation/subgroup_ckd.csv',
]

for rel in checks:
    p = Path(rel)
    print(rel, 'OK' if p.exists() else 'MISSING')


outputs/best_models_info.json OK
outputs/shap/shap_importance.csv OK
outputs/external_validation/external_validation_results.csv OK
outputs/advanced_evaluation/advanced_evaluation_summary.json OK
outputs/advanced_evaluation/calibration_metrics.csv OK
outputs/advanced_evaluation/dca_results.csv OK
outputs/advanced_evaluation/subgroup_gender.csv OK
outputs/advanced_evaluation/subgroup_ckd.csv OK


In [25]:
from pathlib import Path
import shutil

zip_path = Path('/content/hf_risk_final_outputs.zip')
print('Checking:', zip_path)
if not zip_path.exists():
    raise FileNotFoundError('Zip not found in /content. Run the ZIP creation cell first.')

print('Zip exists. Size MB:', round(zip_path.stat().st_size / 1_000_000, 2))

# 1) Try browser download to local PC
download_ok = False
try:
    from google.colab import files
    files.download(str(zip_path))
    download_ok = True
    print('Download was triggered. Check your browser Downloads folder.')
except Exception as exc:
    print('Browser download did not trigger:', exc)

# 2) Always keep a persistent backup in Google Drive if available
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    drive_target = Path('/content/drive/MyDrive/hf_risk_final_outputs.zip')
    shutil.copy2(zip_path, drive_target)
    print('Backup copied to Google Drive:', drive_target)
except Exception as exc:
    print('Drive backup skipped:', exc)

if not download_ok:
    print('Use Colab Files pane to manually download /content/hf_risk_final_outputs.zip')

Mounted at /content/drive
Backup copied to Google Drive: /content/drive/MyDrive/hf_risk_final_outputs.zip


In [24]:
from pathlib import Path
p = Path("/content/hf_risk_final_outputs.zip")
print("Exists in /content:", p.exists())
if p.exists():
    print("Size MB:", round(p.stat().st_size/1_000_000, 2))

Exists in /content: True
Size MB: 9.03
